# Notebook for consolidating test data performance on best models

In [1]:
#!git clone https://github.com/laukamkit/capstone_project_GroupA.git

In [2]:
# %cd capstone_project_GroupA
# !git checkout main

In [3]:
# %cd src

In [4]:
# if you put this into another folder within src, u need below.
# import sys, os
# sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import os
from scipy import stats
import glob
from NSWData.NSWDataLoader import *
from ModelFiles.ModelPlots import *
nsw_data_loader = NSWDataLoader()

Found NSW data path: C:\Users\kyim1\Desktop\capstone_project_GroupA\data\NSW


In [5]:
sarimax_folder = os.path.join(nsw_data_loader.output_dir, 'SARIMAX_results')
metrics_files = glob.glob(os.path.join(sarimax_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
sarimax_results_df = pd.concat(dfs, ignore_index=True)

gb_folder = os.path.join(nsw_data_loader.output_dir, 'gradient_boosting_results')
metrics_files = glob.glob(os.path.join(gb_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
gb_results_df = pd.concat(dfs, ignore_index=True)

lstm_folder = os.path.join(nsw_data_loader.output_dir, 'LSTM_results')
metrics_files = glob.glob(os.path.join(lstm_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
lstm_results_df = pd.concat(dfs, ignore_index=True)

patchtst_folder = os.path.join(nsw_data_loader.output_dir, 'PATCHTST_results')
metrics_files = glob.glob(os.path.join(patchtst_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
patchtst_results_df = pd.concat(dfs, ignore_index=True)

In [6]:
sarimax_results_df['model'] = 'SARIMAX'
sarimax_results_df['seed'] = '31415'
sarimax_results_df = sarimax_results_df[['model', 'seed', 'horizon','rmse','mae']]
gb_results_df['model'] = 'Gradient Boosting'
gb_results_df = gb_results_df[['model', 'seed', 'horizon','rmse','mae']]
lstm_results_df['model'] = 'LSTM'
lstm_results_df = lstm_results_df[['model', 'seed', 'horizon','rmse','mae']]
patchtst_results_df['model'] = 'PATCHTST'
patchtst_results_df = patchtst_results_df[['model', 'seed', 'horizon','rmse','mae']]
all_results_df = pd.concat([sarimax_results_df, gb_results_df, lstm_results_df, patchtst_results_df], ignore_index=True)

In [7]:
all_results_df = all_results_df.groupby(['model', 'horizon'])[['rmse','mae']].agg(['mean', 'std', 'size']).reset_index()
t_crit = all_results_df[('rmse', 'size')].apply(lambda n: stats.t.ppf(0.975, df=n - 1))
se = all_results_df[('rmse', 'std')] / all_results_df[('rmse', 'size')] ** 0.5
all_results_df[('rmse', 'CI_lower')] = all_results_df[('rmse', 'mean')] - t_crit * se
all_results_df[('rmse', 'CI_upper')] = all_results_df[('rmse', 'mean')] + t_crit * se
se = all_results_df[('mae', 'std')] / all_results_df[('mae', 'size')] ** 0.5
all_results_df[('mae', 'CI_lower')] = all_results_df[('mae', 'mean')] - t_crit * se
all_results_df[('mae', 'CI_upper')] = all_results_df[('mae', 'mean')] + t_crit * se
all_results_df[[('rmse','mean'),('rmse','std'),('rmse','CI_lower'),('rmse','CI_upper'),('mae','mean'),('mae','std'),('mae','CI_lower'),('mae','CI_upper')]] = all_results_df[[('rmse','mean'),('rmse','std'),('rmse','CI_lower'),('rmse','CI_upper'),('mae','mean'),('mae','std'),('mae','CI_lower'),('mae','CI_upper')]].round(2)
all_results_df.to_csv(os.path.join(nsw_data_loader.output_dir, 'model_comparison_results.csv'), index=False)
all_results_df

model horizon    rmse                 mae              \
                                 mean    std size    mean    std size   
0   Gradient Boosting      48  536.34   0.01    6  380.49   0.01    6   
1   Gradient Boosting     336  655.67   0.12    6  471.72   0.16    6   
2   Gradient Boosting     720  690.82   0.17    6  498.17   0.17    6   
3                LSTM      48  564.98  20.86    6  402.08  13.51    6   
4                LSTM     336  790.94  17.10    6  574.38  16.85    6   
5                LSTM     720  858.99  13.24    6  635.97  10.64    6   
6            PATCHTST      48  468.07   3.87    6  308.63   2.72    6   
7            PATCHTST     336  637.35  15.00    6  445.23  14.99    6   
8            PATCHTST     720  694.99  19.41    6  496.41  18.38    6   
9             SARIMAX      48  544.65    NaN    1  367.25    NaN    1   
10            SARIMAX     336  650.79    NaN    1  454.65    NaN    1   
11            SARIMAX     720  667.58    NaN    1  471.88    NaN    1   

       rmse               mae           
   CI_lower CI_upper CI_lower CI_upper  
0    536.33   536.35   380.48   380.51  
1    655.54   655.80   471.55   471.89  
2    690.65   691.00   497.99   498.34  
3    543.09   586.88   387.90   416.27  
4    772.99   808.88   556.70   592.06  
5    845.10   872.88   624.81   647.13  
6    464.01   472.13   305.77   311.48  
7    621.61   653.09   429.50   460.96  
8    674.62   715.36   477.11   515.70  
9       NaN      NaN      NaN      NaN  
10      NaN      NaN      NaN      NaN  
11      NaN      NaN      NaN      NaN